# CURE-Rec — end-to-end quickstart

This notebook runs the complete first implementation milestone: configuration and data loading, CURE-Sim creation, exact 64-coalition evaluation, Shapley/interaction regions, robust improvement selection, and structured-log inspection.

**Important:** CURE-Sim is an oracle benchmark. The optional CSV audit below intentionally does not turn an arbitrary interaction file into causal evidence.

## 1. Setup

Run from `paper-ideas/CURE-Rec/code/` after installing `pip install -e '.[dev]'`. The cell also makes the notebook usable directly from a source checkout.

In [8]:
from pathlib import Path
import sys

ROOT = Path.cwd().resolve()
if ROOT.name == 'notebooks':
    ROOT = ROOT.parent
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from cure_rec.config import load_settings
from cure_rec.data import audit_interactions, load_curesim, load_interactions_csv
from cure_rec.pipeline import run_experiment

print('Project root:', ROOT)

Project root: /Users/mlouhichi/Desktop/CURE-Rec/next-paper/paper-ideas/CURE-Rec/code


## 2. Load the experiment configuration and synthetic data-generating environment

`CURE-Sim` is the primary data source for this milestone because it exposes known policy effects, feedback dynamics, and oracle coalition values. The configuration is part of the run manifest.

In [9]:
config_path = ROOT / 'configs' / 'curesim_quickstart.yaml'
settings = load_settings(config_path)
print('Config hash:', settings.config_hash())
print('Users / items / horizon:', settings.simulator.n_users, settings.simulator.n_items, settings.simulator.horizon)
print('Interventions:', list(settings.interventions.costs))
print('Scenarios:', [scenario.name for scenario in settings.scenarios])

# Explicit synthetic data loading: inspect the disclosed starting state.
simulator = load_curesim(settings)
print('Synthetic catalogue shape:', simulator.catalog.features.shape)
print('Synthetic public-profile shape:', simulator.state.public_profiles.shape)


Config hash: a15ab769c9aab911
Users / items / horizon: 24 72 4
Interventions: ['repeat_cap', 'explore_slot', 'tail_slot', 'diversify', 'novel_slot', 'provider_balance']
Scenarios: ['nominal', 'fatigue_stress', 'popularity_stress']
Synthetic catalogue shape: (72, 6)
Synthetic public-profile shape: (24, 6)


## 3. Run the exact intervention game

This evaluates all 64 intervention coalitions under every configured scenario. The output includes:

- full-game Shapley regions;
- feasibility-aware semivalue sensitivity;
- Grabisch–Roubens interaction regions;
- direct robust improvement selection with abstention;
- JSONL event logs and CSV artifacts.

In [10]:
logger, game, decision = run_experiment(settings)
RUN_DIR = logger.run_dir
print('Run directory:', RUN_DIR)
print('Decision:', decision.action)
print('Selected portfolio:', decision.selected_interventions)
print('Worst-case improvement:', round(decision.lower_improvement, 5))

2026-08-04 13:42:09,485 | INFO | run_started | {"config_hash": "a15ab769c9aab911", "run_id": "curesim-quickstart-20260804T124209Z-5700b599"}
2026-08-04 13:42:09,485 | INFO | exact_game_started | {}
2026-08-04 13:42:09,486 | INFO | simulator_ready | {"horizon": 4, "n_items": 72, "n_users": 24, "scenario": "nominal"}
2026-08-04 13:42:14,948 | INFO | scenario_game_completed | {"grand_coalition_improvement": -0.4210904357912536, "scenario": "nominal", "shapley_efficiency_gap": 5.551115123125783e-17}
2026-08-04 13:42:14,949 | INFO | simulator_ready | {"horizon": 4, "n_items": 72, "n_users": 24, "scenario": "fatigue_stress"}
2026-08-04 13:42:20,531 | INFO | scenario_game_completed | {"grand_coalition_improvement": -0.4187732306444988, "scenario": "fatigue_stress", "shapley_efficiency_gap": 5.551115123125783e-17}
2026-08-04 13:42:20,532 | INFO | simulator_ready | {"horizon": 4, "n_items": 72, "n_users": 24, "scenario": "popularity_stress"}
2026-08-04 13:42:26,166 | INFO | scenario_game_comple

## 4. Inspect exact attribution and interaction outputs

In [11]:
display(game.regions.sort_values('phi_mean', ascending=False))
display(game.interaction_table.sort_values('interaction_mean', ascending=False))

,intervention,phi_lower,phi_upper,phi_mean,psi_feasible_lower,psi_feasible_upper,phi_psi_sign_agree
0,repeat_cap,0.019328,0.021651,0.020355,0.019107,0.021458,True
3,diversify,-0.058360,-0.057684,-0.057936,-0.057792,-0.056812,True
2,tail_slot,-0.079257,-0.078018,-0.078605,-0.080382,-0.078750,True
4,novel_slot,-0.092236,-0.087646,-0.089787,-0.093722,-0.088783,True
1,explore_slot,-0.099465,-0.098286,-0.098737,-0.100854,-0.099472,True
5,provider_balance,-0.116113,-0.114098,-0.115354,-0.116218,-0.114792,True


,intervention_i,intervention_j,interaction_lower,interaction_upper,interaction_mean
14,novel_slot,provider_balance,0.002719,0.004190,0.003328
12,diversify,novel_slot,0.002373,0.003395,0.002879
5,explore_slot,tail_slot,0.001386,0.003722,0.002804
7,explore_slot,novel_slot,0.002158,0.003382,0.002772
3,repeat_cap,novel_slot,0.000554,0.001605,0.001177
6,explore_slot,diversify,0.000240,0.001268,0.000872
10,tail_slot,novel_slot,-0.000909,0.001235,0.000504
9,tail_slot,diversify,-0.000561,0.000884,0.000109
2,repeat_cap,diversify,-0.000787,-0.000456,-0.000576
8,explore_slot,provider_balance,-0.001096,-0.000162,-0.000596


## 5. Inspect the direct robust portfolio table

The planner selects by scenario-wise worst-case improvement, not by adding lower Shapley endpoints.

In [12]:
coalitions = game.coalition_table.groupby('mask', as_index=False).agg(
    lower_improvement=('improvement', 'min'),
    upper_improvement=('improvement', 'max'),
    cost=('cost', 'first'),
    interventions=('active_interventions', 'first'),
).sort_values('lower_improvement', ascending=False)
display(coalitions.head(12))

,mask,lower_improvement,upper_improvement,cost,interventions
1,1,0.027033,0.032587,0.05,repeat_cap
0,0,0.000000,0.000000,0.00,
9,9,-0.032752,-0.027082,0.11,repeat_cap;diversify
5,5,-0.055457,-0.052778,0.13,repeat_cap;tail_slot
8,8,-0.060242,-0.058317,0.06,diversify
17,17,-0.067962,-0.064039,0.13,repeat_cap;novel_slot
4,4,-0.077633,-0.073657,0.08,tail_slot
3,3,-0.079463,-0.073913,0.15,repeat_cap;explore_slot
33,33,-0.092927,-0.089431,0.17,repeat_cap;provider_balance
16,16,-0.096463,-0.092377,0.08,novel_slot


## 6. Inspect structured logs and artifacts

The event stream is JSONL, so it is inspectable with Pandas, `jq`, or a text editor. Per-coalition records contain metrics, timing, active interventions, and transform statistics.

In [13]:
import json
import pandas as pd

events_path = RUN_DIR / 'logs' / 'events.jsonl'
events = pd.DataFrame([json.loads(line) for line in events_path.read_text().splitlines()])
display(events[['timestamp_utc', 'event']].tail(12))

for artifact in sorted((RUN_DIR / 'artifacts').glob('*.json')):
    print('artifact:', artifact.name)
for figure in sorted((RUN_DIR / 'figures').glob('*.png')):
    print('figure:', figure.name)

,timestamp_utc,event
399,2026-08-04T12:42:26.173801+00:00,portfolio_rejected
400,2026-08-04T12:42:26.174128+00:00,portfolio_rejected
401,2026-08-04T12:42:26.174374+00:00,portfolio_rejected
402,2026-08-04T12:42:26.174783+00:00,portfolio_rejected
403,2026-08-04T12:42:26.175182+00:00,portfolio_rejected
404,2026-08-04T12:42:26.175500+00:00,portfolio_selected
405,2026-08-04T12:42:26.176621+00:00,explanation_card_written
406,2026-08-04T12:42:26.176824+00:00,robust_planning_completed
407,2026-08-04T12:42:26.177087+00:00,reporting_started
408,2026-08-04T12:42:26.271872+00:00,figures_written


artifact: explanation_card.json
artifact: game_manifest.json
artifact: portfolio_decision.json
artifact: run_summary.json
figure: coalition_improvements.png
figure: shapley_regions.png


## 7. Optional: audit a local interaction CSV

Set `LOCAL_CSV` to a local file containing at least `user_id`, `item_id`, `timestamp`, and `response`. The audit labels the strongest claim supported by the columns; it does not infer missing propensities or exposure fields.

In [14]:
LOCAL_CSV = None  # e.g. ROOT / 'data' / 'raw' / 'interactions.csv'
if LOCAL_CSV is not None:
    frame = load_interactions_csv(LOCAL_CSV)
    audit = audit_interactions(frame)
    print(audit)
else:
    print('No local CSV selected; synthetic CURE-Sim run above is complete.')

No local CSV selected; synthetic CURE-Sim run above is complete.


## 8. Next run

After the quickstart passes, switch to `configs/curesim_full.yaml`, inspect `runs/<run-id>/artifacts/explanation_card.json`, and only then begin audited real-log integration.